In [0]:
import logging
import json
import requests
from tenacity import retry, stop_after_attempt, wait_exponential

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("eia_ingestion")

API_KEY = dbutils.secrets.get(scope="eia_api", key="eia-api-key")
volume_path = "/Volumes/dbr_dev_ua5816bd/roksolana_shendiu770/raw_files/"

In [0]:
CONSUMPTION_URL = "https://api.eia.gov/v2/petroleum/cons/wpsup/data/"
PRICES_URL = "https://api.eia.gov/v2/petroleum/pri/gnd/data/"

@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=2, min=2, max=8))
def fetch_page(url, offset, length=5000, extra_params=None):
    params = {
        "frequency": "weekly",
        "data[0]": "value",
        "start": "2009-01-01",
        "end": "2026-08-24",
        "sort[0][column]": "period",
        "sort[0][direction]": "desc",
        "offset": offset,
        "length": length,
        "api_key": API_KEY,
    }
    if extra_params:
        params.update(extra_params)
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    return response


def fetch_all(url, extra_params=None, page_size=5000):
    all_data = []
    offset = 0
    total = None

    while total is None or offset < total:
        response = fetch_page(url, offset=offset, length=page_size, extra_params=extra_params)
        body = response.json()["response"]

        if total is None:
            total = int(body["total"])
            logger.info(f"Total rows available: {total}")

        all_data.extend(body["data"])
        offset += page_size
        logger.info(f"Fetched {len(all_data)}/{total} rows so far")

    return all_data

In [0]:
consumption_data = fetch_all(CONSUMPTION_URL)
consumption_path = volume_path + "petroleum_raw.json"
with open(consumption_path, "w") as f:
    json.dump(consumption_data, f)
logger.info(f"Saved {len(consumption_data)} consumption rows to {consumption_path}")

prices_data = fetch_all(PRICES_URL, extra_params={"facets[duoarea][]": "NUS"})
prices_path = volume_path + "petroleum_prices_raw.json"
with open(prices_path, "w") as f:
    json.dump(prices_data, f)
logger.info(f"Saved {len(prices_data)} price rows to {prices_path}")